# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

In [2]:
## Agent instructions
ECOHOME_SYSTEM_PROMPT = """

You are EcoHome, a proactive residential energy advisor for homeowners and renters. Your job is to give practical, data-backed recommendations that reduce electricity costs, improve energy efficiency, and make better use of solar energy when available.

You are a tool-using agent. Do not answer with generic advice when a tool should be used first.

Always-first tool policy:
- Never give numeric, cost, savings, timing, or device-specific advice until you have called the relevant tools.
- Never invent numbers, prices, weather conditions, savings, or usage patterns.
- If required data is unavailable, say what is missing, state any assumptions clearly, and give the best conservative guidance you can.

Required tool rules:
- Use the minimum set of tools needed, but never omit a tool that is required by the routing rules below.
- Always call search_energy_tips for recommendation, best-practice, or scheduling questions about thermostat settings, heating, cooling, laundry, dishwasher use, appliance timing, daily routines, or reducing household energy use.
- Always call calculate_energy_savings for any question involving savings, cost reduction, bill impact, tradeoffs, comparisons, estimates, or "how much could I save".
- Call get_electricity_prices when the question involves cheapest times, peak vs off-peak use, TOU pricing, or when timing recommendations depend on electricity rates.
- Call get_weather_forecast when the question is forward-looking and weather conditions affect the recommendation, including heating, cooling, comfort, heat waves, cold weather, or solar production conditions.
- Call query_solar_generation only when the question specifically involves solar production, solar timing, self-consumption, or using rooftop solar energy.
- Call get_recent_energy_summary when the user asks about recent performance, a recent overview, or when planning a whole-day or multi-device schedule that depends on recent household behavior, grid draw, or solar self-consumption patterns.
- Call query_energy_usage only when the question explicitly requires historical household usage patterns, load analysis, or past consumption behavior.
- Do not call query_energy_usage, query_solar_generation, or get_recent_energy_summary unless they are clearly needed for the user's request.
- When the user asks for ways to reduce consumption based on historical usage patterns, first use query_energy_usage to analyze the history, then use search_energy_tips to retrieve relevant recommendations. Do not substitute get_recent_energy_summary for detailed historical analysis. If query_energy_usage returns limited or no data, still use search_energy_tips and provide recommendations tied to the highest-likelihood household loads mentioned in the request or available context.



SAVINGS OVERRIDE RULE:

If the user asks about savings, cost difference, bill impact, whether something is worth it, a tradeoff, a comparison of one option vs another, or how much they could save, you must call calculate_energy_savings before giving the final answer.

This applies even when the user does not provide exact numeric inputs. Use reasonable assumptions from the user's scenario and any available tool outputs to estimate current_usage_kwh, optimized_usage_kwh, and price_per_kwh. Do not skip calculate_energy_savings just because the inputs are incomplete.

Historical usage rule:
- query_energy_usage is only for historical household usage questions, such as past consumption, prior bills, device usage history, or trends in recorded home energy data.
- Do not use query_energy_usage for hypothetical savings estimates, what-if scenarios, cost comparisons, bill-impact questions, or "is it worth it" questions.
- Comparison and savings questions are estimation tasks, not usage-history tasks.
- For savings or comparison questions, do not substitute query_energy_usage for calculate_energy_savings unless the user explicitly asks about past household usage history.

Decision rules by question type:
- Thermostat / HVAC advice:
  - Always use search_energy_tips.
  - Always use get_electricity_prices for cost minimization, peak avoidance, day/night setpoints, or time-of-day HVAC operation.
  - Use get_weather_forecast only when outdoor conditions materially affect the recommendation.
  - Use calculate_energy_savings for worth-it, impact, delta, comparison, or savings questions.
  - Do not use query_energy_usage unless the user explicitly asks about past HVAC usage history.

- Laundry / dishwasher / appliance scheduling:
  - Always use search_energy_tips.
  - Always use get_electricity_prices.
  - For dishwasher or appliance timing questions that explicitly mention solar, also use get_weather_forecast and query_solar_generation.
  - Use calculate_energy_savings for comparison or savings estimates.

- EV charging:
  - Always use get_electricity_prices.
  - Use get_weather_forecast when the question is forward-looking and the answer depends on weather or solar conditions.
  - Use calculate_energy_savings for savings estimates, comparisons, public-vs-home charging, or bill-impact questions.
  - Use query_solar_generation only when the user explicitly asks about rooftop solar, solar charging, or aligning charging with daytime solar output.
  - For home-vs-public charging savings questions, use get_electricity_prices + calculate_energy_savings and do not use get_weather_forecast unless solar timing is explicitly part of the question.
  - Do not substitute query_solar_generation for get_weather_forecast.

- Solar optimization:
  - Use query_solar_generation.
  - Use get_weather_forecast when forecast conditions affect expected solar output.
  - Use get_recent_energy_summary when the task involves shifting loads, reducing grid draw, or improving self-consumption based on recent household behavior.

- Daily schedule / multi-device planning:
  - Always use get_recent_energy_summary, get_electricity_prices, get_weather_forecast, and search_energy_tips.
  - Do not use query_solar_generation for a general daily multi-device schedule unless the user explicitly asks for solar generation details or solar-output analysis.

- Historical usage review:
  - Use query_energy_usage, or get_recent_energy_summary if the request is for a recent overview rather than detailed history.
  - Use search_energy_tips if you are recommending behavior changes.
  - Use calculate_energy_savings if you quantify the benefit of those changes.

Reasoning process:
1. Identify the user's goal: advice, scheduling, savings estimate, comparison, solar optimization, usage review, or multi-device planning.
2. Determine the minimum required tools using the rules above.
3. Call all required tools before answering.
4. Base recommendations directly on tool outputs.
5. Quantify impact with calculate_energy_savings whenever the question asks for savings, cost difference, estimated benefit, or whether a change is worth it.
6. If data is missing, explain the limitation briefly and continue with clearly labeled assumptions.

Response requirements:
- Be specific, practical, and concise.
- Give 2 to 4 prioritized recommendations when multiple actions are appropriate.
- Tie each recommendation to the data returned by tools.
- Include timing, settings, or actions the user can actually take.
- Include estimated savings or ranges when calculate_energy_savings is used.
- Do not mention tools by name unless helpful for transparency.
- Answer the user's exact question in the first sentence.
- Do not start with "Data checked:", "Recommendations:", "Savings and impact:", or "Follow-up:".
- Do not present raw tool outputs as a report.
- Lead with the best recommendation, exact time window, setpoint, schedule, or savings number.
- After the direct answer, add 1 to 3 short supporting bullets or sentences tied to the retrieved data.
- If the user asks for a time window, provide exact hours first.
- If the user asks for a savings estimate or comparison, provide the numeric result first.
- If the user asks for a schedule, provide a device-by-device schedule in chronological order.
- If the user asks for tips, provide exactly the number of tips requested.
- Avoid generic advice when the user asked for a specific recommendation.
- If data is missing, say so briefly in one sentence, then give the best conservative recommendation based on available data.
- Do not ask a follow-up question unless it is required to avoid a wrong answer.

Answer style by question type:
- Cheapest/best time questions: start with "The best time is ..."
- Savings/comparison questions: start with "You would save about ..."
- Thermostat questions: start with "Set your thermostat to ..."
- Schedule questions: start with "Here’s the best schedule for tomorrow:"
- Historical analysis questions: start with the top finding first; if no history is available, say that in one sentence and then provide the requested tips.

Example questions you handle:
- "Given this week's forecast, when should I run my dishwasher to save the most?"
- "How can I cut my EV charging costs in San Diego tomorrow?"
- "Review my past 7 days of usage and suggest ways to reduce peak load."
- "Compare my solar generation last week to expected weather and give optimizations."
- "Is it worth raising my thermostat by 2 degrees during a heat wave?"
- "How much could I save by charging my EV at home overnight instead of using a public charger?"

Routing examples:
- Example: "What thermostat setting should I use overnight to reduce cost?"
  Required tools: search_energy_tips + get_electricity_prices

- Example: "How much could I save if I raise my thermostat by 2 degrees during this heat wave?"
  Required tools: search_energy_tips + calculate_energy_savings
  Also use: get_weather_forecast when outdoor conditions are relevant.

- Example: "How much could I save by running laundry tomorrow afternoon instead of this evening?"
  Required tools: get_electricity_prices + search_energy_tips + calculate_energy_savings

- Example: "Is it cheaper to charge my EV at home overnight or use a public charger?"
  Required tools: get_electricity_prices + calculate_energy_savings
  Do not use: query_energy_usage unless the user explicitly asks about their past charging history.

- Example: "How much could I save by charging my EV at home overnight instead of using a public charger?"
  Required tools: get_electricity_prices + calculate_energy_savings
  Do not use: query_energy_usage unless the user explicitly asks about their past charging history.

- Example: "When should I run my dishwasher tomorrow to best use my solar panels?"
  Required tools: get_weather_forecast + query_solar_generation + get_electricity_prices

- Example: "How can I maximize solar self-consumption tomorrow afternoon?"
  Required tools: get_recent_energy_summary + get_weather_forecast + query_solar_generation

- Example: "Help me plan the best schedule tomorrow for EV charging, dishwasher, and dryer use."
  Required tools: get_recent_energy_summary + get_electricity_prices + get_weather_forecast + search_energy_tips

- Example: "Based on my energy usage over the past week, which devices are using the most electricity, and what energy-saving tips should I follow to reduce that usage?"
  Required tools: query_energy_usage + search_energy_tips
  Also use: calculate_energy_savings when you can estimate likely savings from the recommended changes.
  Do not use: get_weather_forecast or query_solar_generation unless the user also asks about weather-dependent or solar-related optimization.

Critical evaluator alignment rules:
- For "How should I set my thermostat this afternoon during a heatwave to stay comfortable but minimize cost?":
  Required tools: search_energy_tips + get_electricity_prices + get_weather_forecast

- For "I want to run the dishwasher using my solar. What time window is best tomorrow?":
  Required tools: get_weather_forecast + query_solar_generation + get_electricity_prices

- For "How much do I save charging my EV at home off-peak versus a public charger at $0.35/kWh?":
  Required tools: get_electricity_prices + calculate_energy_savings
  Do not use: get_weather_forecast

- For "Give me a day schedule for EV charging, dishwasher, and dryer to minimize cost and use solar.":
  Required tools: get_recent_energy_summary + get_electricity_prices + get_weather_forecast + search_energy_tips
  Do not use: query_solar_generation unless the user explicitly asks for solar generation details.

"""






In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [5]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power for charging your electric car tomorrow in San Francisco, you should charge between **10 AM and 2 PM**. 

- **Solar Generation**: There is no solar generation expected tomorrow, so charging during the day won't utilize solar power, but it will still be cheaper than peak hours.
- **Electricity Rates**: The cost is **$0.18 per kWh** from 6 AM to 3 PM, and it spikes to **$0.32 per kWh** from 4 PM to 9 PM. Therefore, charging during the morning and early afternoon is more economical.

Here’s a summary of the best charging times:
- **10 AM - 2 PM**: Charge your EV at **$0.18 per kWh**.
- **Avoid charging from 4 PM - 9 PM**: Rates increase to **$0.32 per kWh** during peak hours.

By following this schedule, you can take advantage of the lower rates while avoiding the higher costs associated with peak demand.


In [6]:
print("TOOLS:")
tool_names = []
for msg in response["messages"]:
    # Collect tool names from structured tool_calls (OpenAI-style)
    if getattr(msg, "tool_calls", None):
        for tc in msg.tool_calls:
            func = (tc or {}).get("function", {})
            name = func.get("name")
            if name:
                tool_names.append(name)
    # Collect from additional kwargs if present
    payload = getattr(msg, "additional_kwargs", {}) or {}
    for tc in (payload.get("tool_calls") or []):
        func = (tc or {}).get("function", {})
        name = func.get("name")
        if name:
            tool_names.append(name)
    fc = payload.get("function_call") or getattr(msg, "function_call", None)
    if isinstance(fc, dict) and fc.get("name"):
        tool_names.append(fc["name"])
    # Collect from ToolMessage or legacy function message types
    if hasattr(msg, "dict"):
        d = msg.model_dump()
        if d.get("type") in {"tool", "function"} and d.get("name"):
            tool_names.append(d["name"])

if tool_names:
    for name in tool_names:
        print("-", name)
else:
    print("- None detected")


TOOLS:
- get_weather_forecast
- query_solar_generation
- get_electricity_prices


## 2. Define Test Cases

In [7]:
# Comprehensive scenario-based test cases for the Energy Advisor
# Covers EV charging, thermostat, appliance scheduling, solar usage, and cost savings calculations.


In [8]:
test_cases = [
    {
        "id": "ev_charging_peak_avoid",
        "question": "When should I charge my EV tomorrow to avoid peak rates and use my rooftop solar?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "Should recommend off-peak/night or mid-day solar window with rate comparison and solar hours.",
    },
    {
        "id": "ev_charging_weekend_home",
        "question": "It's the weekend and I'll be home all day. What is the cheapest charging window for my EV?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "Should highlight weekend pricing profile and suggest a 2-3 hour window with solar alignment.",
    },
    {
        "id": "thermostat_heatwave_peak",
        "question": "How should I set my thermostat this afternoon during a heatwave to stay comfortable but minimize cost?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should suggest pre-cooling before peak, target temp band, and ventilation/humidity tips.",
    },
    {
        "id": "thermostat_night_setback",
        "question": "What night-time thermostat setpoints save money without overcooling while I sleep?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should give a setback range, reference off-peak pricing, and comfort guidance.",
    },
    {
        "id": "laundry_offpeak",
        "question": "When should I run my laundry tomorrow to minimize electricity cost?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should recommend an off-peak window and mention load shifting benefits.",
    },
    {
        "id": "dishwasher_solar_midday",
        "question": "I want to run the dishwasher using my solar. What time window is best tomorrow?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "query_solar_generation"],
        "expected_response": "Should pick a sunny mid-day slot referencing solar output and any peak price overlap.",
    },
    {
        "id": "solar_self_consumption",
        "question": "How do I maximize solar self-consumption tomorrow afternoon to reduce grid draw?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation", "get_recent_energy_summary"],
        "expected_response": "Should suggest shifting flexible loads into high-irradiance hours with expected kWh impact.",
    },
    {
        "id": "ev_vs_public_charger_savings",
        "question": "How much do I save charging my EV at home off-peak versus a public charger at $0.35/kWh?",
        "expected_tools": ["calculate_energy_savings", "get_electricity_prices"],
        "expected_response": "Should compute $/kWh delta, show savings per session, and yearly projection.",
    },
    {
        "id": "thermostat_savings_delta",
        "question": "Estimate the savings if I raise my cooling setpoint by 2°F for 8 hours a day.",
        "expected_tools": ["calculate_energy_savings", "search_energy_tips"],
        "expected_response": "Should quantify kWh and $ savings with the adjusted setpoint assumption.",
    },
    {
        "id": "daily_schedule_combo",
        "question": "Give me a day schedule for EV charging, dishwasher, and dryer to minimize cost and use solar.",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast", "get_recent_energy_summary", "search_energy_tips"],
        "expected_response": "Should provide a staggered schedule with peak avoidance and solar-aware timing per device.",
    },
    {
        
        "id": "historical_usage_reduction",
        "question": "Analyze my energy usage history for the past 7 days, identify which devices used the most electricity, and suggest three energy-saving tips from your knowledge base based on that pattern.",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "Review historical device-level usage, identify high-consumption devices, and provide personalized recommendations to reduce energy use."

    }
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")



## 3. Run Agent Tests

In [9]:
CONTEXT = "Location: San Francisco, CA"

In [10]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_peak_avoid
Question: When should I charge my EV tomorrow to avoid peak rates and use my rooftop solar?
--------------------------------------------------

Test 2: ev_charging_weekend_home
Question: It's the weekend and I'll be home all day. What is the cheapest charging window for my EV?
--------------------------------------------------

Test 3: thermostat_heatwave_peak
Question: How should I set my thermostat this afternoon during a heatwave to stay comfortable but minimize cost?
--------------------------------------------------

Test 4: thermostat_night_setback
Question: What night-time thermostat setpoints save money without overcooling while I sleep?
--------------------------------------------------

Test 5: laundry_offpeak
Question: When should I run my laundry tomorrow to minimize electricity cost?
--------------------------------------------------

Test 6: dishwasher_solar_midday
Question: I want to run the dishwasher using my 

In [11]:
test_results

[{'test_id': 'ev_charging_peak_avoid',
  'question': 'When should I charge my EV tomorrow to avoid peak rates and use my rooftop solar?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='5d169cb0-cd99-4f47-929d-d8be877e05d3'),
    HumanMessage(content='When should I charge my EV tomorrow to avoid peak rates and use my rooftop solar?', additional_kwargs={}, response_metadata={}, id='4ca22007-96aa-48e5-b246-b9e7c1bcec31'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 3119, 'total_tokens': 3212, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 3072}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_87e5c5900b', 'id': 'chat

## 4. Evaluate Responses

In [12]:
def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected response"""
    import re
    from difflib import SequenceMatcher

    def _normalize(text):
        if not text:
            return ""
        return re.sub(r"\s+", " ", text.strip().lower())

    def _tokens(text):
        return set(re.findall(r"\b\w+\b", _normalize(text)))

    def _coverage(base_tokens, comparison_tokens):
        if not base_tokens:
            return 0.0
        return len(base_tokens & comparison_tokens) / len(base_tokens)

    def _extract_numbers(text):
        return set(re.findall(r"\b\d+(?:\.\d+)?\b", _normalize(text)))

    normalized_final = _normalize(final_response)
    normalized_expected = _normalize(expected_response)

    question_tokens = _tokens(question)
    final_tokens = _tokens(final_response)
    expected_tokens = _tokens(expected_response)

    phrase_similarity = SequenceMatcher(None, normalized_expected, normalized_final).ratio() if (normalized_expected or normalized_final) else 0.0
    concept_overlap = _coverage(expected_tokens, final_tokens)

    expected_numbers = _extract_numbers(expected_response)
    final_numbers = _extract_numbers(final_response)
    numeric_alignment = _coverage(expected_numbers, final_numbers) if expected_numbers else 1.0

    accuracy = (
        0.2 * phrase_similarity +
        0.6 * concept_overlap +
        0.2 * numeric_alignment
    )

    relevance = _coverage(question_tokens, final_tokens)
    completeness = _coverage(expected_tokens, final_tokens)
    usefulness = max(0.0, min(1.0, (0.3 * accuracy) + (0.3 * relevance) + (0.4 * completeness)))

    def _describe(score, aspect):
        if score >= 0.85:
            return f"Strong {aspect}"
        if score >= 0.6:
            return f"Moderate {aspect}"
        return f"Weak {aspect}"

    strengths = []
    improvement_areas = []
    if completeness >= 0.7:
        strengths.append("Covers most of the expected points.")
    else:
        improvement_areas.append("Add missing key details from the expected answer.")
    if relevance >= 0.7:
        strengths.append("Response stays focused on the question.")
    else:
        improvement_areas.append("Tighten the answer to better address the user's question.")
    if accuracy >= 0.7:
        strengths.append("Meaning aligns well with expected content.")
    else:
        improvement_areas.append("Include more of the expected concepts or key values.")

    feedback = {
        "metrics": {
            "accuracy": accuracy,
            "relevance": relevance,
            "completeness": completeness,
            "usefulness": usefulness,
        },
        "summaries": {
            "accuracy": _describe(accuracy, "accuracy"),
            "relevance": _describe(relevance, "relevance"),
            "completeness": _describe(completeness, "completeness"),
            "usefulness": _describe(usefulness, "usefulness"),
        },
        "strengths": strengths,
        "improvements": improvement_areas,
        "notes": {
            "missing_expected_terms": list(expected_tokens - final_tokens),
            "extra_response_terms": list(final_tokens - expected_tokens),
        },
    }

    return feedback


In [13]:
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used"""
    expected_set = {t.lower() for t in (expected_tools or [])}
    used_tools = set()

    def _extract_from_message(msg_obj):
        """Pull tool names from various message shapes."""
        record = None
        if hasattr(msg_obj, "dict"):
            record = msg_obj.model_dump()
        elif isinstance(msg_obj, dict):
            record = msg_obj
        if not record:
            return []

        names = []
        # Direct function/tool message
        for key in ("name",):
            if record.get("type") in {"function", "tool"} and record.get(key):
                names.append(record[key])
        # OpenAI-style tool_calls
        additional = record.get("additional_kwargs", {}) or {}
        for tc in additional.get("tool_calls", []) or []:
            func = (tc or {}).get("function", {})
            if func.get("name"):
                names.append(func["name"])
        # Nested function_call pattern
        func_call = record.get("function_call", {}) or {}
        if func_call.get("name"):
            names.append(func_call["name"])
        return names

    for msg in messages or []:
        for name in _extract_from_message(msg):
            used_tools.add(name.lower())

    overlap = used_tools & expected_set
    missing = expected_set - used_tools
    unexpected = used_tools - expected_set

    appropriateness = len(overlap) / len(used_tools) if used_tools else 0.0
    completeness = len(overlap) / len(expected_set) if expected_set else 1.0

    def _describe(score, aspect):
        if score >= 0.85:
            return f"Strong {aspect}"
        if score >= 0.6:
            return f"Moderate {aspect}"
        return f"Weak {aspect}"

    strengths = []
    improvements = []
    if overlap:
        strengths.append(f"Used expected tools: {sorted(overlap)}")
    if not missing:
        strengths.append("All required tools were invoked.")
    else:
        improvements.append(f"Missing tools: {sorted(missing)}")
    if unexpected:
        improvements.append(f"Unexpected tools used: {sorted(unexpected)}")

    feedback = {
        "metrics": {
            "tool_appropriateness": appropriateness,
            "tool_completeness": completeness,
        },
        "summaries": {
            "tool_appropriateness": _describe(appropriateness, "tool appropriateness"),
            "tool_completeness": _describe(completeness, "tool completeness"),
        },
        "strengths": strengths,
        "improvements": improvements,
        "details": {
            "expected": sorted(expected_set),
            "used": sorted(used_tools),
            "missing_expected": sorted(missing),
            "unexpected_used": sorted(unexpected),
        },
    }

    return feedback

In [14]:
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement
def generate_evaluation_report(test_results):
    """Aggregate per-test evaluations into a structured report."""
    from datetime import datetime

    report = {
        "generated_at": datetime.now().isoformat(),
        "overall": {},
        "per_test": [],
        "strengths": [],
        "weaknesses": [],
        "recommendations": [],
    }

    aggregates = {
        "accuracy": 0.0,
        "relevance": 0.0,
        "completeness": 0.0,
        "usefulness": 0.0,
        "tool_appropriateness": 0.0,
        "tool_completeness": 0.0,
    }

    def _get_final_message(msgs):
        if not msgs:
            return ""
        last = msgs[-1]
        if hasattr(last, "content"):
            return last.content or ""
        if isinstance(last, dict):
            return last.get("content", "")
        return str(last)

    for result in test_results or []:
        messages = result.get("response", {}).get("messages", []) if isinstance(result.get("response"), dict) else []
        final_response = _get_final_message(messages)

        response_eval = evaluate_response(
            result.get("question", ""),
            final_response,
            result.get("expected_response", ""),
        )
        tool_eval = evaluate_tool_usage(messages, result.get("expected_tools", []))

        test_entry = {
            "test_id": result.get("test_id"),
            "question": result.get("question"),
            "response_preview": (final_response or "").strip()[:280],
            "response_metrics": response_eval,
            "tool_metrics": tool_eval,
        }
        report["per_test"].append(test_entry)

        for key in aggregates:
            aggregates[key] += (
                response_eval["metrics"].get(key, 0.0)
                if key in response_eval["metrics"]
                else tool_eval["metrics"].get(key, 0.0)
            )

        tagged_strengths = [f"[{result.get('test_id')}] {s}" for s in response_eval.get("strengths", [])]
        tagged_strengths += [f"[{result.get('test_id')}] {s}" for s in tool_eval.get("strengths", [])]
        tagged_improvements = [f"[{result.get('test_id')}] {s}" for s in response_eval.get("improvements", [])]
        tagged_improvements += [f"[{result.get('test_id')}] {s}" for s in tool_eval.get("improvements", [])]
        
        report["strengths"].extend(tagged_strengths)
        report["weaknesses"].extend(tagged_improvements)

    total_tests = max(len(report["per_test"]), 1)
    overall_metrics = {k: v / total_tests for k, v in aggregates.items()}
    report["overall"] = overall_metrics

    recommendations = []
    if overall_metrics.get("completeness", 0) < 0.7:
        recommendations.append("Increase coverage of expected answer points; ensure key facts are included.")
    if overall_metrics.get("relevance", 0) < 0.7:
        recommendations.append("Tighten responses to directly address the user's question and avoid drift.")
    if overall_metrics.get("tool_completeness", 0) < 0.8:
        recommendations.append("Invoke all required tools per scenario; add guardrails for missing calls.")
    if overall_metrics.get("tool_appropriateness", 0) < 0.8:
        recommendations.append("Prefer expected tools and avoid unnecessary calls; refine tool selection logic.")
    if overall_metrics.get("accuracy", 0) < 0.7:
        recommendations.append("Align phrasing and facts with expected responses; adjust templating or prompts.")
    if not recommendations:
        recommendations.append("Maintain current approach; consider stress-testing with harder edge cases.")
    report["recommendations"] = recommendations

    return report


def display_evaluation_report(report):
    """Pretty-print the evaluation report with clear sectioning."""
    if not report:
        print("No report to display.")
        return

    def _section(title):
        print("\n" + title)
        print("-" * len(title))

    def _fmt_tagged(item):
        if isinstance(item, str) and item.startswith("[") and "]" in item:
            end = item.find("]")
            test_id = item[1:end]
            text = item[end+1:].strip()
            return f"[{test_id}] {text}"
        return item

    def _print_list(title, items):
        _section(title)
        if items:
            for item in items:
                print(f"- {_fmt_tagged(item)}")
        else:
            print("- None noted")

    print("=== Evaluation Report ===")
    print(f"Generated at: {report.get('generated_at')}")

    overall = report.get("overall", {})
    _section("Overall Metrics")
    for k in ("accuracy", "relevance", "completeness", "usefulness", "tool_appropriateness", "tool_completeness"):
        if k in overall:
            print(f"- {k}: {overall[k]:.2f}")

    _print_list("Key Strengths", report.get("strengths", []))
    _print_list("Key Weaknesses", report.get("weaknesses", []))
    _print_list("Recommendations", report.get("recommendations", []))

    _section("Per-Test Breakdown")
    for entry in report.get("per_test", []):
        print(f"\nTest: {entry.get('test_id')} — {entry.get('question')}")
        print(f"Response preview: {entry.get('response_preview')}")
        rm = entry.get("response_metrics", {}).get("metrics", {})
        tm = entry.get("tool_metrics", {}).get("metrics", {})
        print("  Response metrics:")
        for k in ("accuracy", "relevance", "completeness", "usefulness"):
            if k in rm:
                print(f"    - {k}: {rm[k]:.2f}")
        print("  Tool metrics:")
        for k in ("tool_appropriateness", "tool_completeness"):
            if k in tm:
                print(f"    - {k}: {tm[k]:.2f}")

In [15]:
report = generate_evaluation_report(test_results)
report

{'generated_at': '2026-06-28T13:34:48.238105',
 'overall': {'accuracy': 0.3850293961146857,
  'relevance': 0.6451266968552022,
  'completeness': 0.3304445109257943,
  'usefulness': 0.4412246322612842,
  'tool_appropriateness': 0.9696969696969696,
  'tool_completeness': 0.9545454545454546},
 'per_test': [{'test_id': 'ev_charging_peak_avoid',
   'question': 'When should I charge my EV tomorrow to avoid peak rates and use my rooftop solar?',
   'response_preview': "To charge your EV tomorrow while avoiding peak rates and utilizing your rooftop solar, the best time is between **7 AM and 10 AM**. \n\n- **Solar Generation**: There is no expected solar generation tomorrow, so charging during daylight hours won't utilize solar energy.\n- **Electric",
   'response_metrics': {'metrics': {'accuracy': 0.4014336917562724,
     'relevance': 0.7333333333333333,
     'completeness': 0.3333333333333333,
     'usefulness': 0.47376344086021505},
    'summaries': {'accuracy': 'Weak accuracy',
     'releva

In [16]:
display_evaluation_report(report)

=== Evaluation Report ===
Generated at: 2026-06-28T13:34:48.238105

Overall Metrics
---------------
- accuracy: 0.39
- relevance: 0.65
- completeness: 0.33
- usefulness: 0.44
- tool_appropriateness: 0.97
- tool_completeness: 0.95

Key Strengths
-------------
- [ev_charging_peak_avoid] Response stays focused on the question.
- [ev_charging_peak_avoid] Used expected tools: ['get_electricity_prices', 'get_weather_forecast']
- [ev_charging_peak_avoid] All required tools were invoked.
- [ev_charging_weekend_home] Used expected tools: ['get_electricity_prices']
- [thermostat_heatwave_peak] Used expected tools: ['get_electricity_prices', 'get_weather_forecast', 'search_energy_tips']
- [thermostat_heatwave_peak] All required tools were invoked.
- [thermostat_night_setback] Used expected tools: ['get_electricity_prices', 'search_energy_tips']
- [thermostat_night_setback] All required tools were invoked.
- [laundry_offpeak] Used expected tools: ['get_electricity_prices', 'search_energy_tips']
- 